# Reto: Auditoría energética exprés de un edificio de oficinas

### Aplica lo que aprendiste en EDA y Machine Learning — tú solo(a), con Copilot como apoyo

---

## El encargo

Un edificio de oficinas les entrega **45 días de mediciones horarias** (archivo `reto_edificio_oficinas.csv`, en la misma carpeta que este notebook) y les hace un encargo corto: una primera auditoría energética exprés, con datos y con un modelo, para presentar en una reunión de una hora.

Ya no hay nadie guiándolos celda por celda. Aquí ustedes deciden **qué hacer, en qué orden, y cuánto código escribir a mano vs. pedirle a Copilot**. Este notebook solo les da la estructura (las preguntas que hay que responder y el tiempo sugerido para cada una) — el análisis lo construyen ustedes.

## Datos disponibles

El archivo `reto_edificio_oficinas.csv` tiene una fila por hora, con estas columnas:

| Columna | Qué representa |
|---|---|
| `fecha_hora` | Momento de la medición |
| `energia_kwh` | Energía consumida en esa hora |
| `demanda_kw` | Potencia demandada en ese instante |
| `temperatura_c` | Temperatura ambiente |
| `ocupacion` | Número de personas en el edificio |
| `tipo_dia` | "Laboral" o "Fin de semana" |
| `estado_hvac` | Si el sistema de climatización estaba "Encendido" o "Apagado" |

> No asuman que estos datos llegan limpios. Como en la clase de EDA, es su responsabilidad auditarlos antes de sacar cualquier conclusión.

## Las cuatro preguntas que deben responder

1. **¿Podemos confiar en estos datos?** (calidad)
2. **¿Cuándo y por qué consume energía este edificio?** (patrones y relaciones)
3. **¿Se puede predecir el consumo a partir de ocupación y temperatura?** (modelo)
4. **¿Qué le recomendarían al administrador del edificio?** (conclusión)

## Cómo van a trabajar

- Pueden apoyarse en **GitHub Copilot** para escribir código, tal como practicaron en la clase anterior: formulen el prompt con nombres exactos de columnas y el resultado que esperan, y **validen** lo que les devuelva antes de darlo por bueno.
- Al final de este notebook hay un **banco de pistas opcional** (prompts de referencia), por si algún equipo se atasca. Intenten resolver primero sin mirarlo.
- Este notebook trae únicamente **celdas de tarea** (con instrucciones) y celdas de código **vacías** para que ustedes trabajen. No hay respuestas escondidas.

## Cronómetro sugerido (60 minutos)

| Tarea | Minutos |
|---|---|
| 1. Cargar y explorar los datos | 8 |
| 2. Auditoría rápida de calidad | 7 |
| 3. Limpieza mínima necesaria | 8 |
| 4. Patrones de consumo (mínimo 2 gráficos) | 12 |
| 5. Modelo de predicción de consumo | 15 |
| 6. Conclusión ejecutiva | 10 |

## Entregables (lo que deben tener listo al final)

- [ ] Una tabla de auditoría de calidad, con al menos completitud y validez revisadas.
- [ ] Una versión limpia de los datos, con las decisiones de limpieza justificadas en una línea de texto.
- [ ] Al menos **dos visualizaciones** que respondan la pregunta 2.
- [ ] Un modelo entrenado y evaluado (con una métrica de error) que responda la pregunta 3.
- [ ] Una conclusión ejecutiva de máximo 6 líneas, con al menos una recomendación concreta.

## Tarea 0 · Preparar el entorno (ya resuelta)

Esta celda sí viene dada, para que no pierdan tiempo del reto en pasos administrativos. Ejecútenla y empiecen a trabajar desde la Tarea 1.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

df = pd.read_csv("reto_edificio_oficinas.csv", parse_dates=["fecha_hora"])
print("Filas cargadas:", len(df))
df.head()

Filas cargadas: 1083


,fecha_hora,energia_kwh,demanda_kw,temperatura_c,ocupacion,tipo_dia,estado_hvac
0,2026-04-01 00:00:00,5.42,5.64,20.20,3,Laboral,Apagado
1,2026-04-01 01:00:00,6.45,6.57,16.98,6,Laboral,Apagado
2,2026-04-01 02:00:00,9.91,9.84,17.18,4,Laboral,Apagado
3,2026-04-01 03:00:00,8.24,9.17,17.49,3,Laboral,Apagado
4,2026-04-01 04:00:00,4.96,5.42,16.19,0,Laboral,Apagado


## Tarea 1 · Exploración inicial — 8 minutos

Antes de nada, confirmen que entienden la estructura básica de la tabla.

**Qué deben lograr en esta celda:**

- Ver las dimensiones de la tabla (filas y columnas).
- Confirmar el periodo de tiempo que cubren los datos.
- Revisar el tipo de dato de cada columna (`.info()`), prestando atención a `fecha_hora` — ¿quedó como fecha o como texto?

💡 *Si necesitan ayuda con el código, pídanselo a Copilot describiendo exactamente qué necesitan ver, con el nombre de la variable `df`.*

In [ ]:
# TODO: inspeccionen dimensiones, periodo de tiempo y tipos de dato del DataFrame df.

## Tarea 2 · Auditoría rápida de calidad — 7 minutos

Recuerden las cuatro dimensiones de calidad que vimos en la clase de EDA: **completitud, unicidad, validez y consistencia**. No tienen que hacer una auditoría exhaustiva — es un reto corto — pero sí deben poder responder, con evidencia (una tabla o un número), estas tres preguntas:

1. ¿Qué columnas tienen valores faltantes, y cuántos?
2. ¿Hay marcas de tiempo (`fecha_hora`) duplicadas?
3. ¿Los valores de `energia_kwh` son todos físicamente posibles (no negativos)? ¿Y la columna `estado_hvac` tiene categorías escritas de forma consistente?

In [ ]:
# TODO: construyan una tabla o unos prints que respondan las tres preguntas de calidad.

## Tarea 3 · Limpieza mínima necesaria — 8 minutos

Con lo que encontraron en la Tarea 2, hagan **solo** la limpieza que sea necesaria — no hace falta repetir todo el proceso extenso de la clase de EDA. Como mínimo:

- Eliminen filas con `fecha_hora` duplicada (si las hay).
- Decidan qué hacer con los valores de `energia_kwh` físicamente imposibles (¿los convierten en faltante? ¿los interpolan?).
- Unifiquen la escritura de `estado_hvac` si encontraron inconsistencias.

Guarden el resultado en una nueva variable, por ejemplo `df_limpio`, para no perder el DataFrame original por si necesitan volver a consultarlo.

> Escriban, en una celda de texto (Markdown) después de su código, **una línea explicando cada decisión de limpieza que tomaron y por qué**. Esa justificación es parte del entregable.

In [ ]:
# TODO: limpien lo estrictamente necesario y guarden el resultado en df_limpio.

*(Escriban aquí, en una o dos líneas, qué decisiones de limpieza tomaron y por qué.)*

## Tarea 4 · Patrones de consumo — 12 minutos

Con `df_limpio`, respondan la pregunta 2 del encargo: **¿cuándo y por qué consume energía este edificio?**

Construyan **al menos dos** visualizaciones (elijan las que mejor respondan la pregunta; no hace falta hacer las cinco). Algunas ideas, no obligatorias:

- Un perfil horario de consumo promedio (como el de la clase de EDA).
- Una comparación de consumo entre días laborales y fines de semana.
- Una relación entre `ocupacion` o `temperatura_c` y `energia_kwh` (dispersión o correlación).
- Un mapa de calor de consumo por hora y tipo de día.

Después de cada gráfico, escriban **una frase de interpretación**: ¿qué les dice ese gráfico sobre el edificio?

In [ ]:
# TODO: primera visualización.

*(Interpretación de la primera visualización, en una frase.)*

In [ ]:
# TODO: segunda visualización.

*(Interpretación de la segunda visualización, en una frase.)*

## Tarea 5 · Modelo de predicción de consumo — 15 minutos

Ahora respondan la pregunta 3: **¿se puede predecir el consumo a partir de ocupación y temperatura?**

Sigan el mismo patrón que en la clase de Machine Learning:

1. Elijan las variables de entrada (como mínimo `ocupacion` y `temperatura_c`; pueden agregar más si quieren, como la hora del día).
2. Dividan los datos en entrenamiento y prueba, **respetando el orden cronológico** (no aleatorio).
3. Entrenen un modelo de regresión (el que prefieran: regresión lineal o bosque aleatorio).
4. Evalúen con al menos una métrica (error promedio o R²) sobre el conjunto de prueba.
5. Si usaron un bosque aleatorio, revisen qué variable resultó más importante.

💡 *Si usan una variable de texto como `tipo_dia` o `estado_hvac` como entrada, recuerden que primero hay que codificarla (como hicimos con `tarifa` en la clase anterior).*

In [ ]:
# TODO: preparar variables de entrada (codificar categorías si las usan).

In [ ]:
# TODO: dividir en entrenamiento y prueba de forma cronológica.

In [ ]:
# TODO: entrenar el modelo y evaluarlo con al menos una métrica.

## Tarea 6 · Conclusión ejecutiva — 10 minutos

Cierren el reto con una conclusión de **máximo 6 líneas**, dirigida al administrador del edificio (que no sabe de estadística ni de machine learning). Debe incluir, como mínimo:

- Un hecho respaldado por un número (por ejemplo, el error de su modelo, o el porcentaje de consumo fuera de horario).
- Una interpretación de ese hecho.
- **Una recomendación concreta y accionable.**
- Una limitación honesta de su análisis (por ejemplo: "solo 45 días de datos", "no incluye el clima de todas las estaciones", etc.).

Escriban su conclusión reemplazando el texto de la celda siguiente.

*(Escriban aquí su conclusión ejecutiva, máximo 6 líneas.)*

---

## 🆘 Banco de pistas opcional — solo si su equipo se atasca

No lo lean antes de intentar cada tarea por su cuenta. Son prompts de referencia para Copilot, por si se quedan sin ideas de cómo formular la pregunta.

**Para la Tarea 1 (exploración):**
> Tengo un DataFrame `df` con una columna `fecha_hora`. Muéstrame sus dimensiones, el rango de fechas mínimo y máximo, y un resumen de tipos de dato con `.info()`.

**Para la Tarea 2 (calidad):**
> Tengo un DataFrame `df` con columnas `fecha_hora`, `energia_kwh`, `demanda_kw`, `temperatura_c`, `ocupacion`, `tipo_dia`, `estado_hvac`. Constrúyeme una tabla con el porcentaje de valores faltantes por columna, dime cuántas filas tienen `fecha_hora` duplicada, cuántas filas tienen `energia_kwh` negativa, y muéstrame el conteo de valores únicos de `estado_hvac`.

**Para la Tarea 3 (limpieza):**
> Tengo un DataFrame `df` con una columna de fecha `fecha_hora`, una columna numérica `energia_kwh` que no debería tener valores negativos, y una columna de texto `estado_hvac` con inconsistencias de mayúsculas y espacios. Escribe código en pandas que elimine duplicados por `fecha_hora`, convierta los valores negativos de `energia_kwh` en NaN y los rellene por interpolación temporal, y unifique el texto de `estado_hvac` quitando espacios y usando formato título. Guarda el resultado en `df_limpio`.

**Para la Tarea 4 (patrones):**
> Tengo un DataFrame `df_limpio` indexado o con una columna `fecha_hora`, con columnas `energia_kwh`, `ocupacion`, `temperatura_c` y `tipo_dia`. Grafica el consumo promedio de energía por hora del día, separando entre días "Laboral" y "Fin de semana", en el mismo gráfico con dos líneas.

**Para la Tarea 5 (modelo):**
> Tengo un DataFrame `df_limpio` ordenado cronológicamente por `fecha_hora`, con columnas `energia_kwh`, `ocupacion`, `temperatura_c` y `tipo_dia` (texto). Escribe código en Python con scikit-learn que codifique `tipo_dia` con `pd.get_dummies`, divida los datos en 80% entrenamiento y 20% prueba de forma cronológica (sin mezclar aleatoriamente), entrene un `RandomForestRegressor` para predecir `energia_kwh` a partir de `ocupacion`, `temperatura_c` y las columnas codificadas de `tipo_dia`, y calcule el error absoluto promedio y el R² sobre el conjunto de prueba.